In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode
except ImportError:
    !python -m pip install openmdao[notebooks]

# PassthruComp

There are times when using OpenMDAO when it is desirable to have an input turned into an output.
For example, instead of using promotion to connect two inputs together, sometimes it's more convenient to have an output source available.
In very large models, it can be useful to know that something is an output which can be connected to downstream targets rather than needing to promote the target to the same name.

In these cases, using `PassthruComp` may be useful.
`PassthruComp` efficiently copies each input into a corresponding output which can then be connected to targets elsewhere in the model without need of promotion.

## Adding Variables

Efficient operation of `PassthruComp` relies on corresponding inputs and outputs being added simultaneosly.
While the user is able to do this with `add_input` and `add_output`, `PassthruComp` contains the convenience method `add_var` to streamline the process.

```{eval-rst}
    .. automethod:: openmdao.components.passthru_comp.PassthruComp.add_var
        :noindex:
```

### Variable Renaming

By default, an input named `input_name` has a corresponding output named `input_name_value`.
The user is able to override this behavior by specifying an `output_name` in the call to `add_var`.

### Tagging

The `add_var` method allows for three different ways of adding tags.

- `tags` are applied to both inputs and outputs
- `input_tags` are applied to only inputs
- `output_tags` are applied to only inputs

### Other input/output-specific arguments

The `PassthruComp.add_var` method does not accept `copy_shape` or `shape_by_conn` arguments.
These are replaced by `input_copy_shape`, `output_copy_shape`, `input_shape_by_conn`, and `output_shape_by_conn`.

## Example: Converting an input to an explicit output

In this example, a `PassthruComp` is used to turn the input `length` into an output which is then connetect to two downstream inputs.

In [ ]:
import numpy as np
import openmdao.api as om

p = om.Problem()

ptc = p.model.add_subsystem(name='ptc', subsys=om.PassthruComp(), promotes_inputs=['*'])

i_meta, o_meta = ptc.add_var('length', val=0.5, units='m')

p.model.add_subsystem('area_comp',
                      om.ExecComp('area = x ** 2',
                                  x={'units': 'm'},
                                  area={'units': 'm**2'}))

p.model.add_subsystem('volume_comp',
                      om.ExecComp('volume = x ** 3',
                                  x={'units': 'm'},
                                  volume={'units': 'm**3'}))


p.model.connect('ptc.length_value', ('area_comp.x', 'volume_comp.x'))

p.setup()

p.set_val('length', 0.5)

p.run_model()

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(p.get_val('area_comp.area'), 0.5**2)
assert_near_equal(p.get_val('volume_comp.volume'), 0.5**3)

Note the above example can easily be achieved using promotion as well.

However, in large models with various authors it may be difficult to know if a variable is available for connection, or if the target is required to be promoted.

In [ ]:
import numpy as np
import openmdao.api as om

p = om.Problem()

p.model.add_subsystem('area_comp',
                      om.ExecComp('area = x ** 2',
                                  x={'units': 'm'},
                                  area={'units': 'm**2'}),
                     promotes_inputs=[('x', 'length')])

p.model.add_subsystem('volume_comp',
                      om.ExecComp('volume = x ** 3',
                                  x={'units': 'm'},
                                  volume={'units': 'm**3'}),
                     promotes_inputs=[('x', 'length')])

p.setup()

p.set_val('length', 0.5)

p.run_model()

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(p.get_val('area_comp.area'), 0.5**2)
assert_near_equal(p.get_val('volume_comp.volume'), 0.5**3)